# Docker para MLOps — Bella Tavola 🐳
## Parte 1: Construindo e rodando sua primeira imagem

## Como usar este caderno?

Este caderno abre o ciclo Docker da disciplina. Ao final de e04-p03, o pipeline de CI estava verde e o comentário final dizia: *"Próximo passo: configurar deploy automático."* Este caderno começa a responder essa pergunta.

Nesta parte, o foco é local: você vai entender o que é um contêiner, rodar imagens prontas e — o passo central — escrever o `Dockerfile` da API Bella Tavola e rodá-la em um contêiner.

**Nenhuma conta externa é necessária além do Docker instalado.** O trabalho todo acontece no seu terminal e no seu editor, não neste caderno. As células de código são referências para estudar e adaptar — não clique em "executar".

Os gabaritos estão nas células colapsadas logo após cada exercício — tente resolver antes de abrir.

## O que é essencial, recomendado e desafio?

**Essencial:** o mínimo que você precisa conseguir fazer para completar a etapa.

**Recomendado:** amplia a qualidade da solução.

**Desafio 🎯:** aprofunda a solução do ponto de vista de engenharia.

## Pré-requisitos

Antes de começar, confirme que você já tem:

- A API Bella Tavola funcionando localmente (execute e confirme)
- O modelo publicado no Hugging Face Hub e `load_model` funcionando
- O pipeline de CI da Semana 3 verde no GitHub Actions
- Docker instalado (instruções na primeira seção)

### Verificação rápida do ambiente

Antes de seguir, rode cada comando e confirme a saída esperada:

In [ ]:
# Execute no terminal — não aqui

# 1. A API sobe sem erro?
# uvicorn main:app --reload
# Saída esperada: INFO: Application startup complete.

# 2. A API responde?
# curl http://localhost:8000/
# Saída esperada: {"restaurante": "Bella Tavola", ...}

# 3. load_model funciona?
# python -c "from model_utils import load_model; print('OK')"
# Saída esperada: OK

# Se qualquer item falhar, resolva antes de continuar.
# Os Blocos 8 e 9 dependem de uma API que funciona localmente.

---

# BLOCO 7 — Por que Docker? O problema que você já conhece

> **Objetivo:** Conectar Docker ao problema que você já vivenciou — a máquina limpa do CI — e entender o que contêineres resolvem de forma diferente (e complementar) a um ambiente virtual Python.

## Conceitos-chave do Bloco 7

### O problema que você já conheceu

Na Semana 3, você criou um ambiente virtual limpo para simular o CI:

```bash
python -m venv env_ci_test
source env_ci_test/bin/activate
pip install -r requirements.txt
pytest tests/ -v
```

Isso revelou dependências que estavam na sua máquina mas não no `requirements.txt`. A lição foi: o `requirements.txt` é a única fonte de verdade sobre as dependências do projeto.

Mas o ambiente virtual tem um limite. Ele isola **pacotes Python** — e só isso.

### O que um ambiente virtual isola — e o que não isola

| O que o `venv` isola | O que o `venv` **não** isola |
|---|---|
| Pacotes Python (versões exatas) | Versão do Python em si |
| Scripts de entrada-point do pip | Binários do sistema operacional (`curl`, `git`, `libssl`) |
| Conflitos entre projetos Python | Variáveis de ambiente do sistema |
| — | Versão do sistema operacional |
| — | Configurações de rede e portas |

Isso costuma não ser problema em desenvolvimento. Mas imagine:

- Sua API usa `scikit-learn` compilado contra `libgomp` versão X. O servidor de produção tem versão Y. A API carrega — mas prediz errado, silenciosamente.
- Seu código funciona no Python 3.11 que você tem. O servidor tem 3.9. `match/case` levanta `SyntaxError`.
- Sua máquina tem `curl` disponível (usado em um script de startup). A imagem de produção é mínima e não tem.

### O que um contêiner isola

Um contêiner isola **tudo**: o sistema operacional, as bibliotecas do sistema, os binários, as variáveis de ambiente, os processos, a rede. Ele é uma unidade que carrega seu próprio ambiente completo.

A diferença conceitual fundamental:

```
Ambiente virtual:  seu código + pacotes Python  →  roda no SO do host
Contêiner:         seu código + pacotes Python + SO completo  →  roda em isolamento
```

Isso não torna o `venv` obsoleto — em desenvolvimento local, ele continua sendo a ferramenta certa. O contêiner resolve o problema diferente de *"como garantir que o ambiente em produção é idêntico ao ambiente em desenvolvimento"*.

### Imagem vs. contêiner

A distinção mais importante de Docker, e a que mais confunde no início:

| Conceito | Analogia | Descrição |
|---|---|---|
| **Imagem** | Receita de bolo / classe Python | Definição estática e imutável. Descreve *o que existe* no ambiente. Não consome CPU nem memória em repouso. |
| **Contêiner** | Bolo assado / instância Python | Uma imagem em execução. Consome recursos. Pode ser iniciado, parado, removido. Vários contêineres podem rodar a partir da mesma imagem. |

A relação é análoga à de classe e objeto em Python:

```python
# Imagem = classe (definição)
class BellaTavolaAPI:
    ...

# Contêiner = instância (em execução)
api_producao  = BellaTavolaAPI()  # contêiner 1
api_staging   = BellaTavolaAPI()  # contêiner 2 — mesma imagem, instância diferente
```

### Docker Hub

O Docker Hub ([hub.docker.com](https://hub.docker.com)) é o registry público de imagens — análogo ao PyPI para pacotes Python. Quando você escrever `FROM python:3.11-slim` no seu Dockerfile, o Docker vai buscar essa imagem no Hub. Imagens oficiais (Python, PostgreSQL, Nginx) são mantidas pelas próprias organizações e auditadas.

## Exercício 7.1 — Diagnóstico do ambiente atual

**Nível:** Essencial  
**Conceito:** Identificar o que um ambiente virtual não resolve antes de aprender Docker.

Sem escrever nenhum comando Docker ainda, responda as perguntas abaixo nos comentários. O objetivo é tornar explícito o que você vai resolver ao longo deste caderno.

**Para refletir:** pense na API Bella Tavola que você construiu. Ela tem dependências além dos pacotes Python — pense no modelo de ML, no token do Hugging Face, no uvicorn como processo, na porta em que ela ouve. O que um `venv` cobre? O que não cobre?

In [ ]:
# ✏️ Responda nos comentários:

# 1. Quais dependências da API Bella Tavola NÃO são cobertas por um ambiente virtual?
#    (pense além dos pacotes pip)
#    Resposta:
#    O venv cobre dependências Python instaladas via pip.
#    Ele não cobre o sistema operacional,
#    configuração de rede, variáveis de ambiente

# 2. Na Semana 3, quando você criou o ambiente virtual limpo (env_ci_test),
#    o que falhou que não falhava localmente?
#    Resposta:  diferença de configuração, pacotes instalados,
#    problemas de importação

# 3. Se você entregasse só os arquivos .py e o requirements.txt para alguém
#    que nunca viu seu projeto, o que essa pessoa precisaria fazer
#    ALÉM de "pip install -r requirements.txt" para rodar a API?
#    Resposta: Versão correta do python instalado,
#    As variáveis do HF, Comando de como rodar.

<details>
<summary>💡 Gabarito</summary>

```python
# 1. Dependências não cobertas pelo venv:
#    - Versão exata do Python (3.11 vs 3.9 vs 3.12)
#    - Bibliotecas do sistema (libgomp para scikit-learn, libssl para requests)
#    - O binário uvicorn como processo gerenciado
#    - A variável de ambiente HF_TOKEN
#    - A porta 8000 disponível no sistema
#    - Eventual arquivo .env com configurações

# 2. Problemas típicos no env_ci_test:
#    - ModuleNotFoundError para pacotes instalados globalmente mas não no requirements.txt
#    - FileNotFoundError para o arquivo .env (não commitado por segurança)
#    - Falha ao carregar o modelo se HF_TOKEN não estava exportado

# 3. Além de pip install, a pessoa precisaria:
#    - Ter Python 3.11 (não 3.9, não 3.12)
#    - Criar um arquivo .env com HF_TOKEN
#    - Saber que deve rodar uvicorn main:app --host 0.0.0.0 --port 8000
#    - Ter as bibliotecas de sistema necessárias instaladas no SO
#    - Torcer para que o SO seja compatível com as versões compiladas dos pacotes
#
# Docker empacota tudo isso em uma única unidade.
```

</details>

## Exercício 7.2 — Instalação e verificação

**Nível:** Essencial  
**Conceito:** Ter o Docker funcionando e entender o que acontece no primeiro `run`.

### Instalação

- **Windows e macOS:** instale o [Docker Desktop](https://www.docker.com/products/docker-desktop). Ele inclui o Docker Engine, o CLI e uma interface gráfica.
- **Linux:** instale o Docker Engine seguindo as instruções oficiais para sua distribuição em [docs.docker.com/engine/install](https://docs.docker.com/engine/install). Após instalar, adicione seu usuário ao grupo `docker` para não precisar de `sudo`: `sudo usermod -aG docker $USER` (requer logout/login para ter efeito).

### Sua tarefa

1. Instale o Docker conforme seu sistema operacional
2. Execute os comandos de verificação abaixo no terminal
3. Leia a saída do `hello-world` com atenção — ela descreve exatamente o que acabou de acontecer
4. Anote suas observações nos comentários

In [ ]:
# Execute no terminal:

# Verificar versão instalada
# docker --version
# Saída esperada: Docker version 25.x.x, build xxxxxxx

# Rodar o primeiro contêiner
# docker run hello-world

# A saída do hello-world descreve o que aconteceu em 4 passos:
# 1. O Docker CLI enviou o comando para o Docker daemon
# 2. O daemon não encontrou a imagem 'hello-world' localmente
# 3. O daemon fez pull da imagem do Docker Hub
# 4. O daemon criou um contêiner, que imprimiu a mensagem e encerrou

# ✏️ Anote aqui:

# Qual versão do Docker foi instalada?
# Docker version 29.3.0-1

# A mensagem "Unable to find image 'hello-world:latest' locally" apareceu?
# O que isso significa?
# Significa que a imagem não foi encontrada

# O contêiner ainda existe após rodar?
# Verifique com: docker ps -a
# Sim, mas esta como exited(0), ele foi executado e encerrado.

<details>
<summary>💡 Gabarito</summary>

```bash
# Saída esperada do docker run hello-world:

# Unable to find image 'hello-world:latest' locally
# latest: Pulling from library/hello-world
# ...
# Hello from Docker!
# This message shows that your installation appears to be working correctly.
# ...
# To generate this message, Docker took the following steps:
# 1. The Docker client contacted the Docker daemon.
# 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
# 3. The Docker daemon created a new container from that image which runs the
#    executable that produces the output you are currently reading.
# 4. The Docker daemon streamed that output to the Docker client, which sent it
#    to your terminal.

# "Unable to find image locally" significa que o Docker procurou primeiro
# no cache local (imagens já baixadas) e não encontrou.
# Na próxima vez que você rodar hello-world, essa linha NÃO aparece —
# o Docker reutiliza a imagem em cache.

# O contêiner AINDA EXISTE após rodar — mas está parado.
# docker ps -a mostra: STATUS = Exited (0)
# Contêineres parados não desaparecem automaticamente.
# Para remover: docker rm <id>
# Para rodar e já remover ao sair: docker run --rm hello-world
```

</details>

## Checklist do Bloco 7

- [ ] Consigo explicar a diferença entre imagem e contêiner com minhas próprias palavras
- [ ] Consigo explicar o que um contêiner isola além de um ambiente virtual
- [ ] Docker instalado: `docker --version` retorna uma versão
- [ ] `docker run hello-world` funcionou e li a saída com atenção

---

# BLOCO 8 — Primeiros comandos: rodando contêineres

> **Objetivo:** Dominar o ciclo básico de trabalho com Docker — pull, run, ps, stop, rm — e entender o ciclo de vida de um contêiner antes de construir o seu próprio.

## Conceitos-chave do Bloco 8

### O ciclo de vida de um contêiner

```
Imagem no registry  →  pull  →  Imagem local  →  run  →  Contêiner rodando
                                                              ↓
                                              stop  →  Contêiner parado  →  rm  →  Removido
```

Contêineres parados **não desaparecem automaticamente**. Eles ocupam espaço em disco e ficam listados em `docker ps -a`. Isso é útil para inspecionar logs depois que um contêiner encerrou — mas é fácil acumular dezenas deles sem perceber.

### Tabela de comandos essenciais

| Comando | O que faz | Exemplo |
|---|---|---|
| `docker pull <imagem>` | Baixa uma imagem do registry sem rodar | `docker pull python:3.11-slim` |
| `docker run <imagem>` | Cria e inicia um contêiner a partir da imagem | `docker run python:3.11-slim` |
| `docker run -it <imagem> bash` | Abre um shell interativo dentro do contêiner | `docker run -it python:3.11-slim bash` |
| `docker run -d <imagem>` | Roda em background (detached) | `docker run -d nginx` |
| `docker run -p 8000:8000 <imagem>` | Mapeia porta do host para porta do contêiner | `docker run -p 8000:8000 minha-api` |
| `docker run --rm <imagem>` | Remove o contêiner automaticamente ao encerrar | `docker run --rm hello-world` |
| `docker ps` | Lista contêineres **em execução** | — |
| `docker ps -a` | Lista **todos** os contêineres, incluindo parados | — |
| `docker stop <id>` | Para um contêiner graciosamente | `docker stop abc123` |
| `docker rm <id>` | Remove um contêiner parado | `docker rm abc123` |
| `docker images` | Lista imagens locais | — |
| `docker rmi <imagem>` | Remove uma imagem local | `docker rmi python:3.11-slim` |
| `docker logs <id>` | Exibe a saída do contêiner | `docker logs abc123` |
| `docker logs -f <id>` | Acompanha logs em tempo real (follow) | `docker logs -f abc123` |
| `docker exec -it <id> bash` | Abre shell em contêiner já rodando | `docker exec -it abc123 bash` |

### Mapeamento de portas: por que `-p 8000:8000`?

O contêiner tem sua própria interface de rede — isolada do host. A porta 8000 dentro do contêiner não é automaticamente a porta 8000 do seu computador.

```
Seu computador (host)          Contêiner
─────────────────────          ─────────────────────
porta 8000  ←──────── -p 8000:8000 ────────→  porta 8000
porta 8001  ←──────── -p 8001:8000 ────────→  porta 8000  (mesma porta interna, porta externa diferente)
```

O formato é sempre `-p PORTA_HOST:PORTA_CONTAINER`.

### Layers de imagem: intuição

Uma imagem Docker é composta de **camadas** (layers). Cada instrução no `Dockerfile` cria uma camada. Quando o Docker baixa uma imagem, ele baixa apenas as camadas que ainda não tem localmente — isso é por que um segundo `docker pull` da mesma imagem é muito mais rápido.

Você pode ver as camadas de uma imagem com:
```bash
docker history python:3.11-slim
```

## Exercício 8.1 — Explorando o contêiner do Python

**Nível:** Essencial  
**Conceito:** Modo interativo, sistema de arquivos efêmero, diferença entre o ambiente do contêiner e o host.

### Referência

```bash
# Execute na raiz de qualquer diretório — não precisa ser o projeto

# Abre um shell bash dentro de um contêiner Python
docker run -it python:3.11-slim bash

# Saída esperada: você entra no shell do contêiner
# O prompt muda para algo como: root@a3f2b1c9d8e7:/#
# Isso significa que você está DENTRO do contêiner agora.

# Dentro do contêiner, explore:
python --version          # Python 3.11.x
pip list                  # apenas os pacotes mínimos — sem fastapi, sem scikit-learn
cat /etc/os-release       # Debian GNU/Linux — não é o seu macOS ou Ubuntu
whoami                    # root

# Crie um arquivo de teste
echo "olá do contêiner" > /tmp/teste.txt
cat /tmp/teste.txt

# Saia do contêiner
exit

# Saída esperada ao sair: você voltou ao terminal do host
# O prompt voltou ao normal
```

### Sua tarefa

1. Execute os comandos de exploração acima dentro do contêiner
2. Após sair, rode `docker run -it python:3.11-slim bash` novamente
3. Tente acessar o arquivo `/tmp/teste.txt` que você criou
4. Anote o que aconteceu e por quê

In [ ]:
# ✏️ Anote suas observações:

# O arquivo /tmp/teste.txt ainda existia no segundo run?
# Não, pq o arquivo foi criado apenas no primeiro container

# O sistema operacional dentro do contêiner era o mesmo do seu host?
# Não, pq o sistema carrega o seu proprio SO

# O que isso implica para dados que a API Bella Tavola cria
# (registros no banco, arquivos de log)?
# Que os dados podem ser perdidso caso ele termine.

<details>
<summary>💡 Gabarito</summary>

```python
# O arquivo NÃO existia no segundo run.
# Cada 'docker run' cria um contêiner NOVO a partir da imagem.
# A imagem é imutável — nada que você escreve dentro de um contêiner
# volta para a imagem ou aparece em outro contêiner.
# Isso é chamado de efemeridade: o estado do contêiner é descartado ao removê-lo.

# O SO dentro do contêiner era Debian Linux,
# independente de você estar no macOS ou Windows.
# Isso é exatamente o ponto — o contêiner carrega seu próprio SO.

# Implicação para a API Bella Tavola:
# Qualquer dado que a API crie dentro do contêiner (banco SQLite, logs)
# vai DESAPARECER quando o contêiner for removido.
# No Bloco 10 (e05-p02), resolveremos isso com volumes.
# Por enquanto, guarde essa observação — ela vai fazer sentido em breve.
```

</details>

## Exercício 8.2 — Rodando uma API em background

**Nível:** Essencial  
**Conceito:** Modo detached `-d`, mapeamento de portas `-p`, `docker logs`, ciclo stop/rm.

### Referência

```bash
# Execute na raiz de qualquer diretório

# Roda o Nginx (servidor web) em background
docker run -d -p 8080:80 --name meu-nginx nginx

# Saída esperada: um hash longo como
# a3f2b1c9d8e7f6a5b4c3d2e1f0a9b8c7d6e5f4a3b2c1d0e9f8a7b6c5d4e3f2a1
# Isso é o ID do contêiner. O terminal voltou imediatamente — o contêiner roda em background.

# Verificar que está rodando
docker ps
# Saída esperada: uma linha com STATUS = Up X seconds

# Ver os logs do Nginx sem entrar no contêiner
docker logs meu-nginx
# Saída esperada: logs de inicialização do Nginx

# Testar que o servidor responde
curl http://localhost:8080
# Saída esperada: HTML da página padrão do Nginx

# Parar o contêiner
docker stop meu-nginx
# Saída esperada: meu-nginx

# Verificar que parou (mas ainda existe)
docker ps        # não aparece — está parado
docker ps -a     # aparece — STATUS = Exited (0)

# Remover definitivamente
docker rm meu-nginx
```

### Sua tarefa

1. Execute a sequência acima com o Nginx
2. Enquanto o Nginx estiver rodando, abra `http://localhost:8080` no navegador
3. Use `docker logs -f meu-nginx` para acompanhar os logs em tempo real enquanto acessa a página (a flag `-f` é de _follow_). Para sair do follow: `Ctrl+C` — o contêiner **continua rodando**.
4. Pare e remova o contêiner
5. Repita a sequência com `--rm` e observe a diferença:

In [ ]:
# Execute no terminal:

# docker run -d -p 8080:80 --rm --name meu-nginx-temp nginx
# docker stop meu-nginx-temp
# docker ps -a   ← o contêiner aparece?

# ✏️ Anote:

# Com --rm, o contêiner ainda aparecia em 'docker ps -a' após o stop?
# Não, depois do stop, o contêiner 'meu-nginx-temp' não apareceu mais
# em docker ps -a

# Quando faz sentido usar --rm?
# Faz sentido usar --rm em execuções temporárias e  testes rápidos

# Quando NÃO faz sentido usar --rm?
# Analisar logs , ver arquivos gerados, ver falhas

<details>
<summary>💡 Gabarito</summary>

```python
# Com --rm, o contêiner NÃO aparece em docker ps -a após o stop.
# Ele é removido automaticamente quando para.

# Quando usar --rm:
# - Contêineres de uso único (rodar um script, uma migração, um teste)
# - Quando você não precisa inspecionar logs depois que parou
# - Para evitar acúmulo de contêineres parados

# Quando NÃO usar --rm:
# - Quando precisa debugar: se o contêiner parou com erro,
#   você quer poder rodar docker logs <id> depois para entender o que aconteceu
# - Contêineres de longa duração em produção
```

</details>

## Exercício 8.3 — Inspecionando um contêiner em execução

**Nível:** Recomendado  
**Conceito:** `docker exec`, inspecionar o ambiente interno sem parar o contêiner.

### Referência

```bash
# Execute com um contêiner Nginx rodando em background (do exercício anterior)
docker run -d -p 8080:80 --name meu-nginx nginx

# Entra no contêiner SEM pará-lo
docker exec -it meu-nginx bash
# Saída esperada: prompt dentro do contêiner
# root@a3f2b1c9:/

# Dentro do contêiner:
ps aux          # lista processos — veja o processo do Nginx
env             # lista variáveis de ambiente do contêiner
ls /etc/nginx   # arquivos de configuração do Nginx

# Sair do exec sem matar o contêiner
exit
# (ou Ctrl+D)

# Verificar que o contêiner AINDA está rodando após o exit
docker ps
# Saída esperada: meu-nginx ainda aparece com STATUS = Up

# Limpar
docker stop meu-nginx && docker rm meu-nginx
```

### Sua tarefa

Execute a sequência acima. Enquanto dentro do contêiner com `docker exec`, responda:

In [ ]:
# ✏️ Anote o que você observou dentro do contêiner com docker exec:

# Quantos processos o 'ps aux' mostrou?
# O comando não estava disponível dentro do contêiner.
# A imagem nginx é mínima

# Isso é muito ou pouco comparado com o seu sistema operacional?
# O contêiner é muito mais enxuto
# do que o sistema operacional.

# O comando 'exit' matou o contêiner ou só saiu do exec?
# O comando exit apenas encerrou a sessão aberta com docker exec

# Qual seria um caso de uso real para docker exec?
# Verificar arquivos internos, configuração do serviço,
# variáveis de ambiente ou logs sem parar a aplicação.

<details>
<summary>💡 Gabarito</summary>

```python
# O ps aux dentro do contêiner mostra pouquíssimos processos —
# tipicamente 3 a 5 (o próprio Nginx master + workers + o bash do exec).
# No seu SO, ps aux mostra dezenas ou centenas de processos.
# Isso reflete o isolamento: o contêiner vê apenas seus próprios processos.

# exit saiu apenas do shell aberto pelo exec.
# O contêiner continuou rodando — ele só para com docker stop.
# Diferença importante:
#   docker run -it → o processo principal é o bash; exit para o contêiner
#   docker exec -it → abre um processo adicional; exit fecha só ele

# Casos de uso reais para docker exec:
# - Inspecionar logs dentro do contêiner que não são enviados para stdout
# - Verificar se um arquivo de configuração foi copiado corretamente
# - Rodar um comando pontual (ex: criar tabelas no banco) sem parar o serviço
# - Debug: verificar variáveis de ambiente, processos, conectividade de rede
```

</details>

## Exercício 8.4 — Limpeza do ambiente

**Nível:** Recomendado  
**Conceito:** `docker system prune`, `docker system df`, manutenção do ambiente local.

### Referência

```bash
# Execute no terminal

# Ver quanto espaço Docker está usando
docker system df
# Saída esperada: tabela com Images, Containers, Volumes, Build Cache
# e o espaço de cada um

# Remover tudo que não está em uso:
# - contêineres parados
# - imagens sem tag (dangling)
# - redes não utilizadas
# - cache de build
docker system prune
# Saída esperada: pergunta de confirmação, depois lista do que foi removido

# Para remover também imagens que não estão associadas a nenhum contêiner:
docker system prune -a
# ATENÇÃO: isso remove todas as imagens não em uso, incluindo python:3.11-slim.
# O próximo docker run vai ter que baixá-las novamente.
```

### Sua tarefa

1. Rode `docker system df` e anote o espaço atual
2. Rode `docker system prune` (sem `-a`)
3. Rode `docker system df` novamente e compare

In [ ]:
# ✏️ Anote:

# Espaço antes do prune:
# Images: 1.38GB
# Containers: 86.02kB
# Local Volumes: 0B
# Build Cache: 683.5MB

# Espaço depois do prune:
# Images: 1.38GB
# Containers: 81.92kB
# Local Volumes: 0B
# Build Cache: 683.5MB

# O que foi removido?
# Foi removido um contêiner parado

# Por que é perigoso rodar docker system prune -a em uma máquina de CI
# que faz builds frequentes?
# Porque esse comando pode remover imagens e cache úteis para builds futuros.

<details>
<summary>💡 Gabarito</summary>

```python
# docker system prune sem -a remove:
# - Contêineres parados (Exited)
# - Redes não utilizadas por nenhum contêiner
# - Cache de build "dangling" (camadas sem referência)
# NÃO remove imagens com tag (como python:3.11-slim)

# docker system prune -a remove TUDO acima + imagens sem contêiner associado.
# Em uma máquina de CI com builds frequentes, isso seria problemático porque:
# - Cada build começaria baixando todas as imagens base do zero
# - O cache de layers seria destruído a cada limpeza
# - O tempo de cada build aumentaria significativamente
# Na Semana 3, você usou actions/cache@v4 exatamente para evitar esse problema.
```

</details>

## Checklist do Bloco 8

- [ ] Roda contêineres em modo interativo (`-it`) e em background (`-d`)
- [ ] Mapeia portas com `-p` e verifica que a aplicação responde
- [ ] Diferencia `docker ps` (só rodando) de `docker ps -a` (todos)
- [ ] Usa `docker logs` e `docker logs -f` para ler saída de contêiner em background
- [ ] Para contêineres com `docker stop` e remove com `docker rm`
- [ ] Usa `docker exec -it` para entrar em um contêiner sem pará-lo
- [ ] Sabe o que `docker system prune` faz e quando usar com cautela

---

# BLOCO 9 — Dockerfile: contêinerizando a API Bella Tavola

> **Objetivo:** Escrever o primeiro `Dockerfile` real, fazer o build da imagem e rodar a API Bella Tavola em um contêiner.

## Conceitos-chave do Bloco 9

### Anatomia do Dockerfile

Um `Dockerfile` é uma sequência de instruções que descreve como construir uma imagem. Cada instrução cria uma **camada** (layer). O Docker executa essas instruções em ordem, partindo de uma imagem base.

```dockerfile
# Instrução  Argumento
FROM         python:3.11-slim
WORKDIR      /app
COPY         requirements.txt .
RUN          pip install --no-cache-dir -r requirements.txt
COPY         . .
EXPOSE       8000
CMD          ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Cada instrução, explicada

**`FROM python:3.11-slim`**  
A base da imagem. `python:3.11-slim` é uma imagem Debian mínima com Python 3.11 pré-instalado. Alternativas e por que não usá-las aqui:
- `python:3.11` — imagem completa, ~900MB vs ~130MB do slim. Desnecessário para servir a API.
- `python:3.11-alpine` — imagem Alpine (~50MB), mas usa `musl libc` em vez de `glibc`. Pacotes como `scikit-learn` e `numpy` são compilados para `glibc` — eles falham ou exigem compilação local no Alpine. Explore no exercício 9.4.

**`WORKDIR /app`**  
Define o diretório de trabalho dentro do contêiner. Todos os comandos seguintes (`COPY`, `RUN`, `CMD`) operam relativos a `/app`. Sem isso, os arquivos seriam copiados para a raiz do sistema de arquivos (`/`), misturados com arquivos do SO.

**`COPY requirements.txt .`**  
Copia *apenas* o `requirements.txt` para `/app`. O ponto final significa "o `WORKDIR` atual". Por que separado do `COPY . .`? Veja a explicação de cache abaixo.

**`RUN pip install --no-cache-dir -r requirements.txt`**  
Instala as dependências dentro da imagem. `--no-cache-dir` instrui o pip a não guardar o cache de download — reduz o tamanho final da imagem.

**`COPY . .`**  
Copia todo o código da aplicação para `/app`. Feito *depois* do `pip install` — veja o cache abaixo.

**`EXPOSE 8000`**  
Documentação: informa que a aplicação usa a porta 8000. **Não abre a porta automaticamente** — o `-p` no `docker run` é que faz isso. É como um comentário com significado.

**`CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]`**  
O comando padrão executado quando o contêiner inicia. Use a forma de lista JSON (não `CMD uvicorn main:app ...`) para evitar que o processo rode dentro de um shell intermediário.

Por que `--host 0.0.0.0`? O uvicorn por padrão ouve em `127.0.0.1` — um endereço que existe apenas dentro do contêiner. Para que o mapeamento de portas funcione, ele precisa ouvir em `0.0.0.0`, que significa "todas as interfaces de rede" — inclusive a interface virtual que conecta o contêiner ao host.

### A regra de ouro do cache de build

O Docker tem um sistema de cache de layers. Quando você rebuilda uma imagem, o Docker reusa as camadas que não mudaram — **mas invalida todas as camadas após a primeira mudança**.

```
Instrução          Muda quando...              Impacto no cache
────────────────   ─────────────────────────   ─────────────────────────────────
FROM               nova versão do Python       invalida tudo (raro)
COPY requirements  requirements.txt mudou      invalida RUN e COPY seguintes
RUN pip install    requirements.txt mudou      reexecuta (pode levar minutos)
COPY . .           QUALQUER arquivo mudou      invalida apenas CMD
CMD                CMD mudou                   rebuild instantâneo
```

Se você colocasse `COPY . .` *antes* de `RUN pip install`, qualquer alteração em qualquer arquivo Python invalidaria o `pip install` — e cada rebuild baixaria todas as dependências novamente. Separando os dois `COPY`, o `pip install` só é reexecutado quando o `requirements.txt` de fato muda.

### O build context

Quando você executa `docker build -t minha-imagem .`, o `.` no final não é só "aqui está o Dockerfile". É o **build context** — o diretório cujo conteúdo é enviado para o daemon Docker para que as instruções `COPY` possam funcionar.

Isso significa que **todos os arquivos no diretório são enviados ao daemon** — incluindo `__pycache__`, `.git`, `.env`, o `venv`, arquivos `.pkl` grandes. O `.dockerignore` (que veremos em e05-p02) resolve isso.

## Exercício 9.1 — O primeiro Dockerfile

**Nível:** Essencial  
**Conceito:** Escrever, buildar e rodar a API Bella Tavola em contêiner — incluindo dois checkpoints de falha esperada.

### Referência

```dockerfile
# Dockerfile mínimo para uma API FastAPI genérica
# Estude cada instrução antes de adaptar para a Bella Tavola

# Imagem base: Python 3.11 mínimo (Debian slim)
FROM python:3.11-slim

# Diretório de trabalho dentro do contêiner
# Todos os comandos seguintes operam a partir daqui
WORKDIR /app

# Copia APENAS o requirements.txt primeiro
# Motivo: separar a camada de dependências da camada de código
# → pip install só reexecuta quando requirements.txt muda
COPY requirements.txt .

# Instala as dependências dentro da imagem
# --no-cache-dir: não guarda cache do pip na imagem (reduz tamanho)
RUN pip install --no-cache-dir -r requirements.txt

# Copia o código da aplicação DEPOIS das dependências
# Mudanças no código não invalidam o pip install
COPY . .

# Documenta que a aplicação usa a porta 8000
# (não abre a porta — o -p no docker run faz isso)
EXPOSE 8000

# Comando padrão ao iniciar o contêiner
# Forma de lista JSON: não usa shell intermediário
# --host 0.0.0.0: escuta em todas as interfaces (necessário para o -p funcionar)
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

```bash
# Execute na RAIZ do projeto Bella Tavola (onde main.py está)
# Verifique antes: ls deve mostrar main.py e requirements.txt

# Build da imagem
docker build -t bella-tavola:v1 .

# Saída esperada (primeira vez, sem cache):
# [1/5] FROM docker.io/library/python:3.11-slim
# [2/5] WORKDIR /app
# [3/5] COPY requirements.txt .
# [4/5] RUN pip install --no-cache-dir -r requirements.txt
# [5/5] COPY . .
# Successfully built  (ou: writing image sha256:...)
```

### Sua tarefa

**Passo 1:** Crie o arquivo `Dockerfile` na raiz do projeto Bella Tavola, adaptando a referência acima.

**Passo 2:** Faça o build:

```bash
# Na raiz do projeto (onde main.py e Dockerfile estão)
docker build -t bella-tavola:v1 .
```

**Passo 3 — Checkpoint de falha esperada A — porta não mapeada:**

In [ ]:
# Execute no terminal:

# Rode o contêiner SEM -p (sem mapear porta)
# docker run --rm bella-tavola:v1

# Agora tente acessar http://localhost:8000 no navegador ou:
# curl http://localhost:8000

# ✏️ O que aconteceu?
# O contêiner rodou normalmente e o Uvicorn subiu,
# mas a aplicação não ficou acessível pelo host.

# Por que não funcionou, mesmo o contêiner estando rodando?
# Porque a porta 8000 existia apenas dentro do contêiner.

# Para o contêiner: Ctrl+C (ele está em foreground)

**Passo 4 — Checkpoint de falha esperada B — host errado:**

Antes de continuar, edite temporariamente o `CMD` do seu Dockerfile para **remover** o `--host 0.0.0.0`:

```dockerfile
# Versão com bug — para fins de observação
CMD ["uvicorn", "main:app", "--port", "8000"]
```

Rebuilde e rode com `-p`:

In [ ]:
# Execute no terminal:

# Após editar o CMD para remover --host 0.0.0.0:
# docker build -t bella-tavola:sem-host .
# docker run -p 8000:8000 --rm bella-tavola:sem-host

# Tente acessar: curl http://localhost:8000

# ✏️ O que aconteceu desta vez?
# Mesmo com a porta mapeada, a aplicação continuou inacessível externamente.
# No meu teste, o curl retornou erro de conexão/reset.

# O uvicorn subiu? O que os logs mostram?
# Sim, mostrou que subiu em http://127.0.0.1:8000

# Por que a porta mapeada NÃO é suficiente se o uvicorn ouve em 127.0.0.1?
# Porque representa apenas o próprio contêiner, para aceitar conexões externas vindas do host, é necessário usar --host 0.0.0.0.

<details>
<summary>💡 Gabarito — Falha A e Falha B</summary>

```python
# FALHA A — sem -p:
# O contêiner roda normalmente, o uvicorn sobe, a porta 8000 existe DENTRO do contêiner.
# Mas nenhum tráfego do host consegue chegar a ela.
# É como ter um servidor em um quarto fechado sem porta.
# curl retorna: "Failed to connect to localhost port 8000: Connection refused"
# Solução: adicionar -p 8000:8000

# FALHA B — com -p mas sem --host 0.0.0.0:
# O uvicorn sobe, a porta está mapeada, mas o uvicorn está ouvindo em
# 127.0.0.1 (loopback) DENTRO do contêiner.
# 127.0.0.1 dentro do contêiner é o próprio contêiner — não a interface de rede
# que comunica com o host.
# O -p 8000:8000 mapeia a interface de rede do contêiner para o host,
# mas o uvicorn está ouvindo em um endereço diferente.
# É como ter uma porta no quarto, mas o servidor está numa sala interior sem janelas.
# curl retorna: "Failed to connect" ou a conexão trava e dá timeout.
#
# Solução: CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
# 0.0.0.0 significa "ouve em todas as interfaces de rede" —
# incluindo a interface virtual que conecta o contêiner ao host.
```

</details>

**Passo 5 — A versão correta:**

Restaure o CMD com `--host 0.0.0.0`, rebuilde e confirme que tudo funciona:

In [ ]:
# Execute no terminal, na raiz do projeto:

# docker build -t bella-tavola:v1 .
# docker run -p 8000:8000 --rm bella-tavola:v1

# Em outro terminal:
# curl http://localhost:8000/
# Saída esperada: {"restaurante": "Bella Tavola", ...}

# curl http://localhost:8000/pratos
# Saída esperada: lista de pratos em JSON

# Verificar o tamanho da imagem:
# docker images bella-tavola
# Anote o tamanho reportado na coluna SIZE:

# ✏️ Tamanho da imagem bella-tavola:v1:
# 693MB

<details>
<summary>💡 Gabarito — Dockerfile completo para a Bella Tavola</summary>

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

```bash
# Tamanho típico da imagem: entre 800MB e 1.2GB
# (scikit-learn e suas dependências são pesados)
# No Bloco 12 (e05-p02), multi-stage build reduzirá isso significativamente.
```

</details>

## Exercício 9.2 — Observando o cache de build

**Nível:** Essencial  
**Conceito:** Como o cache de layers funciona na prática — e por que a ordem das instruções importa.

### Referência

```bash
# Execute na raiz do projeto (após o primeiro build do 9.1)

# --- Experimento 1: alterar código (main.py) ---

# Edite main.py: mude a mensagem da rota raiz
# Ex: "mensagem": "Bem-vindo à nossa API" → "mensagem": "Olá do contêiner!"

docker build -t bella-tavola:v1 .

# Saída esperada:
# [1/5] FROM ...                    → CACHED  (imagem base não mudou)
# [2/5] WORKDIR /app                → CACHED
# [3/5] COPY requirements.txt .     → CACHED  (requirements.txt não mudou)
# [4/5] RUN pip install ...         → CACHED  ← isso é o que importa
# [5/5] COPY . .                    → rebuild (código mudou)
# Tempo: segundos (não minutos)

# --- Experimento 2: alterar requirements.txt ---

# Adicione uma linha ao requirements.txt:
# requests  (se já estiver, adicione: httpx)

docker build -t bella-tavola:v1 .

# Saída esperada:
# [1/5] FROM ...                    → CACHED
# [2/5] WORKDIR /app                → CACHED
# [3/5] COPY requirements.txt .     → rebuild (arquivo mudou)
# [4/5] RUN pip install ...         → rebuild ← pip install rodou de novo
# [5/5] COPY . .                    → rebuild
# Tempo: minutos (pip baixa pacotes)
```

### Sua tarefa

Execute os dois experimentos e anote as observações:

In [ ]:
# ✏️ Experimento 1 — alterar main.py:

# Quais steps usaram CACHED?
# Os steps WORKDIR /app, COPY requirements.txt e RUN pip install --no-cache-dir -r requirements.txt
# usaram CACHED

# O pip install reexecutou?
# Não

# Quanto tempo levou o build?
# Aproximadamente 1 segundo.

# ✏️ Experimento 2 — alterar requirements.txt:

# A partir de qual step o cache foi invalidado?
# A invalidação começou a partir do momento em que o contexto/copias precisaram ser refeitos.
# No log, o step COPY . . deixou de usar cache.

# Por que o COPY . . também foi invalidado, mesmo que o código não tenha mudado?
# Porque o contexto enviado para o build mudou.

# Quanto tempo levou o build?
# Demorou bem masi que o experimento 1

# ✏️ Para refletir:
# Se você invertesse a ordem e fizesse COPY . . ANTES de COPY requirements.txt,
# o que aconteceria a cada alteração em qualquer arquivo .py?
# A cada alteração em qualquer arquivo do projeto, o Docker invalidaria mais cedo o cache
# deixando os builds mais lentos.

<details>
<summary>💡 Gabarito</summary>

```python
# Experimento 1 — alterar main.py:
# Steps 1 a 4 usaram CACHED — pip install NÃO reexecutou.
# Apenas o COPY . . (step 5) reexecutou, pois o código mudou.
# Tempo típico: 2 a 5 segundos.

# Experimento 2 — alterar requirements.txt:
# Cache invalidado a partir do step 3 (COPY requirements.txt).
# O pip install (step 4) reexecutou mesmo que nenhum arquivo .py tenha mudado.
# O COPY . . (step 5) também reexecutou porque tudo após a invalidação é refeito.
# Tempo típico: 2 a 5 minutos.

# Para refletir:
# Se a ordem fosse COPY . . antes de COPY requirements.txt:
# - Qualquer alteração em qualquer .py invalidaria COPY . .
# - Isso invalidaria RUN pip install logo depois
# - O pip install reexecutaria a CADA alteração de código
# - Em um pipeline de CI com builds frequentes, isso multiplicaria
#   o tempo de build por 10x ou mais.
# A separação dos dois COPYs é uma das práticas mais impactantes do Docker.
```

</details>

## Exercício 9.3 — Nomeando e versionando imagens

**Nível:** Recomendado  
**Conceito:** Convenção de tags, múltiplas tags por build, o problema com `latest`.

### Referência

```bash
# Execute na raiz do projeto

# Convenção: usuario/repositorio:versao
# Para imagens locais, o usuario/repositorio é opcional

# Aplicar múltiplas tags no mesmo build
docker build -t bella-tavola:v1 -t bella-tavola:latest .

# Verificar as tags criadas
docker images bella-tavola
# Saída esperada:
# REPOSITORY     TAG      IMAGE ID       CREATED          SIZE
# bella-tavola   v1       abc123def456   2 minutes ago    1.1GB
# bella-tavola   latest   abc123def456   2 minutes ago    1.1GB
# Observe: mesmo IMAGE ID — são dois nomes para a mesma imagem

# Adicionar uma tag a uma imagem já existente (sem rebuildar)
docker tag bella-tavola:v1 bella-tavola:v1.0.0
docker images bella-tavola
```

### Sua tarefa

1. Execute o build com duas tags (`v1` e `latest`)
2. Observe que as duas tags têm o mesmo IMAGE ID
3. Faça uma pequena alteração no código e rebuilde **apenas com a tag `v2`** (sem `latest`)
4. Execute `docker images bella-tavola` e responda:

In [ ]:
# ✏️ Após buildar v2 sem atualizar latest:

# A tag :latest agora aponta para v1 ou v2?
# para v2

# Por que isso é perigoso em produção?

# Porque um deploy baseado apenas em :latest pode subir uma versão diferente
# da que o time imagina, dificultando rollback e auditoria.

# Qual estratégia de tag é mais confiável para rastreabilidade?
# A estratégia mais confiável é usar tags versionadas,
# como v1, v2, v3.

<details>
<summary>💡 Gabarito</summary>

```python
# :latest continua apontando para v1 — porque o build de v2 não incluiu a tag latest.
# latest é apenas uma tag como qualquer outra — não é "automática" nem "especial".
# Se você não atualizar latest explicitamente, ela fica desatualizada.

# Perigo em produção:
# Um script de deploy que faz docker pull minha-api:latest e depois docker run
# pode estar rodando uma versão antiga sem saber.
# Se dois ambientes (staging e produção) puxam :latest em momentos diferentes,
# podem estar rodando versões diferentes.

# Estratégia mais confiável:
# Usar o hash do commit Git como tag: bella-tavola:sha-abc1234
# Isso garante rastreabilidade total: dada a tag, você sabe exatamente
# qual commit gerou aquela imagem.
# No Bloco 13 (e05-p03), o pipeline de CI fará isso automaticamente
# com ${{ github.sha }}.
```

</details>

## Exercício 9.4 — Desafio: por que Alpine falha para projetos de ML 🎯

**Nível:** Desafio

**Conceito:** Entender na prática por que a imagem menor nem sempre é a escolha certa.

### Sua tarefa

1. Crie uma cópia do seu Dockerfile chamada `Dockerfile.alpine`
2. Substitua a primeira linha por `FROM python:3.11-alpine`
3. Tente buildar:

In [ ]:
# Execute no terminal, na raiz do projeto:
# docker build -f Dockerfile.alpine -t bella-tavola:alpine .

# ✏️ # O build foi bem-sucedido?
# Em projetos com bibliotecas
# mais pesadas ou científicas Alpine possui dificuldades

# Se falhou: qual foi a mensagem de erro?
# ERROR: Could not build wheels for scikit-learn

# O que isso ensina sobre imagens pequenas para projetos de ML?
# Ensina que imagem menor nem sempre significa melhor escolha.
#

<details>
<summary>💡 Gabarito</summary>

```python
# O build provavelmente falhou durante o pip install de scikit-learn ou numpy.
# Mensagem típica:
# ERROR: Could not build wheels for scikit-learn
# error: command '/usr/bin/gcc' failed with exit code 1
# ou:
# note: This error originates from a subprocess, and is likely not a problem with pip.

# Por que falha:
# Alpine usa musl libc, não glibc (GNU C Library).
# Os wheels (pacotes pré-compilados) do PyPI para scikit-learn, numpy e similares
# são compilados para glibc.
# No Alpine, o pip não encontra um wheel compatível e tenta compilar do código-fonte.
# A compilação exige gcc, headers de desenvolvimento, e outras ferramentas
# que não estão na imagem Alpine mínima.
# Resolver isso exige RUN apk add gcc musl-dev linux-headers python3-dev
# antes do pip install — o que anula boa parte do ganho de tamanho.

# Tamanhos típicos:
# python:3.11-slim   →  130MB  (escolha padrão para projetos de ML)
# python:3.11-alpine →   52MB  (mas não funciona sem configuração adicional)
# python:3.11        →  900MB  (desnecessariamente grande)
#
# Para projetos de ML: python:3.11-slim é o equilíbrio certo.
# Alpine vale a pena apenas para aplicações puras de Python sem extensões C.
```

</details>

## Erros comuns neste bloco

| Mensagem ou sintoma | Causa provável | Solução |
|---|---|---|
| `COPY failed: file not found in build context` | O arquivo referenciado no COPY não existe no diretório onde você rodou o build | Confirme que está na raiz do projeto com `ls` e que os arquivos existem |
| `ModuleNotFoundError` ao iniciar a API | Pacote instalado localmente mas não no `requirements.txt` | Adicione o pacote ao `requirements.txt` e rebuilde |
| Contêiner para imediatamente após `docker run` | CMD inválido, erro na aplicação ao iniciar | Rode sem `-d` para ver o erro, ou `docker logs <id>` |
| `connection refused` ao acessar a API | Faltou `-p`, ou `--host 0.0.0.0` está ausente no CMD | Veja os Checkpoints de Falha Esperada do exercício 9.1 |
| Build muito lento em todo rebuild | `COPY . .` antes de `RUN pip install` | Inverta a ordem: primeiro `COPY requirements.txt .`, depois `RUN pip install`, depois `COPY . .` |

## Checklist do Bloco 9

- [ ] O arquivo `Dockerfile` existe na raiz do projeto com as instruções corretas
- [ ] `docker build -t bella-tavola:v1 .` conclui sem erros
- [ ] Entendo por que `requirements.txt` é copiado antes do código
- [ ] Observei o cache em ação: alterar `main.py` não reexecuta o `pip install`
- [ ] `docker run -p 8000:8000 bella-tavola:v1` sobe a API
- [ ] `curl http://localhost:8000/` retorna resposta JSON da Bella Tavola
- [ ] Entendo por que `--host 0.0.0.0` é necessário no CMD

---

## Checklist de pré-requisitos para e05-p02

Antes de avançar para o próximo caderno, confirme **todos** os itens abaixo. Se qualquer um falhar, resolva antes de continuar — os Blocos 10, 11 e 12 constroem diretamente sobre o que foi feito aqui.

- [ ] `docker build -t bella-tavola:v1 .` conclui sem erros na raiz do projeto
- [ ] `docker run -p 8000:8000 --rm bella-tavola:v1` sobe a API
- [ ] `curl http://localhost:8000/` retorna resposta JSON da Bella Tavola
- [ ] `curl http://localhost:8000/pratos` retorna a lista de pratos
- [ ] Você consegue explicar o que cada instrução do seu `Dockerfile` faz
- [ ] Você entende por que a ordem das instruções afeta o cache de build

---

# Mapa do que foi construído

| Arquivo | Criado neste caderno | O que faz |
|---|---|---|
| `Dockerfile` | Bloco 9 | Define como construir a imagem da API Bella Tavola |

---

# Checklist de competências — e05-p01

**Bloco 7 — Fundamentos**
- [ ] Explico a diferença entre imagem e contêiner com minhas próprias palavras
- [ ] Explico o que um contêiner isola além de um ambiente virtual Python
- [ ] Docker instalado e `docker run hello-world` funcionando

**Bloco 8 — Primeiros comandos**
- [ ] Rodo, inspeciono, paro e removo contêineres
- [ ] Uso `docker logs` e `docker logs -f` para ler saída de contêiner em background
- [ ] Mapeio portas com `-p` e verifico que a aplicação responde
- [ ] Entro em um contêiner em execução com `docker exec -it`
- [ ] Sei o que `docker system prune` faz e quando usá-lo com cautela

**Bloco 9 — Dockerfile**
- [ ] Escrevo um Dockerfile com FROM, WORKDIR, COPY, RUN, EXPOSE, CMD
- [ ] Entendo por que `requirements.txt` é copiado antes do código
- [ ] Faço build, leio a saída step a step, identifico hits de cache
- [ ] Rodo a API Bella Tavola em um contêiner e verifico que responde
- [ ] Entendo por que `--host 0.0.0.0` é necessário no CMD

---

## O que vem a seguir

A API está rodando em um contêiner. Mas há dois problemas que você já deve ter notado:

1. O endpoint `/ml/predict` provavelmente falhou — o contêiner não tem o `HF_TOKEN`.
2. Qualquer dado criado via API (novos pratos, pedidos) desaparece quando o contêiner para.

Em **e05-p02**, esses dois problemas são resolvidos: variáveis de ambiente e volumes no Bloco 10, múltiplos serviços com Docker Compose no Bloco 11, e o `Dockerfile` refatorado com boas práticas no Bloco 12.